In [ ]:
import pandas as pd

: 

## Add Coordinates

In [3]:
def dms_to_decimal(dms_str):
    """Convert degrees-minutes-seconds string to decimal degrees"""
    parts = dms_str.split()
    degrees = float(parts[0])
    minutes = float(parts[1])
    seconds = float(parts[2])
    return degrees + minutes/60 + seconds/3600

In [4]:
df_weather_stations = pd.read_csv('../../data/input/weather/kody_stacji.csv', sep=';')
df_weather_stations.drop('LP.', axis=1, inplace=True)

# only Meteorogical Stations
df_weather_stations = df_weather_stations[df_weather_stations['ID'].astype(str).str.startswith(('2', '3'))] 

df_weather_stations['Szerokość geograficzna'] = df_weather_stations['Szerokość geograficzna'].apply(dms_to_decimal)
df_weather_stations['Długość geograficzna'] = df_weather_stations['Długość geograficzna'].apply(dms_to_decimal)


df_weather_stations.drop(columns=['Rzeka'], inplace=True)
df_weather_stations.rename(columns={
    'ID': 'id',
    'Nazwa': 'name',
    'Szerokość geograficzna': 'latitude',
    'Długość geograficzna': 'longitude',
    'Wysokość n.p.m.': 'elevation',
}, inplace=True)

df_weather_stations['id'] = df_weather_stations['id'].astype(str)

df_weather_stations

,id,name,latitude,longitude,elevation
0,249170080,Dzierżkowice,49.992222,17.846389,270.0
1,249180010,Pszczyna,49.995556,18.919167,270.0
2,249180020,Warszowice,49.991944,18.705556,270.0
3,249180030,Chałupki,49.925833,18.327500,198.0
4,249180070,Mazańcowice,49.862500,18.969444,269.0
...,...,...,...,...,...
721,354180135,Hel,54.603611,18.811944,1.0
722,354180155,Gdańsk-Świbno,54.333611,18.934444,7.0
723,354190160,Elbląg-Milejewo,54.223333,19.543611,189.0
724,354210185,KĘTRZYN,54.067222,21.366667,107.0


In [56]:
df_weather_stations.to_csv('../../data/intermediate/weather_stations.csv', index=False)

In [32]:
import os
import pandas as pd
import numpy as np
import re
import glob
from collections import defaultdict

def find_station_start_dates(data_dir, df_weather_stations):
    """
    Find when each weather station first appears in measurements for each parameter.
    Updates the dataframe with start dates for each parameter.
    
    Args:
        data_dir: Root directory containing weather data files
        df_weather_stations: DataFrame containing weather station information
    
    Returns:
        Updated DataFrame with start dates for each parameter
    """
    # Parameters to analyze with their codes
    parameters = {
        'B00300S': 'temperature',
        'B00802A': 'humidity', 
        'B00606S': 'precipitation'
    }
    
    # Dictionary to store earliest appearance date for each station/parameter
    start_dates = defaultdict(dict)
    
    # Scan all data files for each parameter
    for param_code, param_name in parameters.items():
        print(f"Analyzing {param_name} (code: {param_code})...")
        
        # Find all files for this parameter
        all_files = []
        for root, _, files in os.walk(data_dir):
            for file in files:
                if file.startswith(param_code) and file.endswith('.csv'):
                    all_files.append(os.path.join(root, file))
        
        print(f"Found {len(all_files)} files for {param_name}")
        
        # Sort files by date to process chronologically
        all_files.sort()
        
        for file_path in all_files:
            # Extract year and month from filename
            basename = os.path.basename(file_path)
            match_year = re.search(r'_(\d{4})_', basename)
            match_month = re.search(r'_\d{4}_(\d{2})\.csv', basename)
            
            if match_year and match_month:
                year = int(match_year.group(1))
                month = int(match_month.group(1))
                
                try:
                    # Read the data file
                    df = pd.read_csv(
                        file_path, 
                        sep=';', 
                        encoding='windows-1250',
                        names=['station_id', 'measurement', 'datetime', 'value'], 
                        header=None, 
                        index_col=False
                    )
                    
                    # Clean and convert station IDs to integers
                    df['station_id'] = pd.to_numeric(df['station_id'], errors='coerce')
                    df = df.dropna(subset=['station_id'])
                    
                    # For each unique station in this file
                    for station_id in df['station_id'].unique():
                        station_id = int(station_id)
                        
                        # Record this as the start date if it's the first appearance
                        if param_name not in start_dates[station_id]:
                            start_dates[station_id][param_name] = (year, month)
                
                except Exception as e:
                    print(f"Error processing file {file_path}: {e}")
    
    # Display discovered start dates
    print("\nDiscovered station start dates:")
    for station_id, params in start_dates.items():
        for param_name, (year, month) in params.items():
            print(f"Station {station_id}, {param_name}: {year}-{month:02d}")
    
    # Update the DataFrame with start dates
    for station_id in start_dates:
        for param_name in start_dates[station_id]:
            year, month = start_dates[station_id][param_name]
            column_name = f"{param_name}_start_date"
            
            # Create column if it doesn't exist
            if column_name not in df_weather_stations.columns:
                df_weather_stations[column_name] = None
            
            # Update the start date for this station and parameter
            mask = df_weather_stations['id'] == str(station_id)
            if mask.any():
                df_weather_stations.loc[mask, column_name] = f"{year}-{month:02d}"
            else:
                print(f"Warning: Station ID {station_id} not found in weather stations DataFrame")
    
    return df_weather_stations

# Example usage:
df_weather_stations = find_station_start_dates('../../data/input/weather', df_weather_stations)

Analyzing temperature (code: B00300S)...
Found 60 files for temperature


/var/folders/yp/gjwvp9611c96kbtrz5ssff_h0000gn/T/ipykernel_32108/3085457296.py:58: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(
/var/folders/yp/gjwvp9611c96kbtrz5ssff_h0000gn/T/ipykernel_32108/3085457296.py:58: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(
/var/folders/yp/gjwvp9611c96kbtrz5ssff_h0000gn/T/ipykernel_32108/3085457296.py:58: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(
/var/folders/yp/gjwvp9611c96kbtrz5ssff_h0000gn/T/ipykernel_32108/3085457296.py:58: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(
/var/folders/yp/gjwvp9611c96kbtrz5ssff_h0000gn/T/ipykernel_32108/3085457296.py:58: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df 

Analyzing humidity (code: B00802A)...
Found 60 files for humidity


/var/folders/yp/gjwvp9611c96kbtrz5ssff_h0000gn/T/ipykernel_32108/3085457296.py:58: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(
/var/folders/yp/gjwvp9611c96kbtrz5ssff_h0000gn/T/ipykernel_32108/3085457296.py:58: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(
/var/folders/yp/gjwvp9611c96kbtrz5ssff_h0000gn/T/ipykernel_32108/3085457296.py:58: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(
/var/folders/yp/gjwvp9611c96kbtrz5ssff_h0000gn/T/ipykernel_32108/3085457296.py:58: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(
/var/folders/yp/gjwvp9611c96kbtrz5ssff_h0000gn/T/ipykernel_32108/3085457296.py:58: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df 

Analyzing precipitation (code: B00606S)...
Found 60 files for precipitation


/var/folders/yp/gjwvp9611c96kbtrz5ssff_h0000gn/T/ipykernel_32108/3085457296.py:58: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(
/var/folders/yp/gjwvp9611c96kbtrz5ssff_h0000gn/T/ipykernel_32108/3085457296.py:58: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(
/var/folders/yp/gjwvp9611c96kbtrz5ssff_h0000gn/T/ipykernel_32108/3085457296.py:58: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(
/var/folders/yp/gjwvp9611c96kbtrz5ssff_h0000gn/T/ipykernel_32108/3085457296.py:58: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(
/var/folders/yp/gjwvp9611c96kbtrz5ssff_h0000gn/T/ipykernel_32108/3085457296.py:58: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df 


Discovered station start dates:
Station 249190890, temperature: 2015-01
Station 249190890, humidity: 2015-01
Station 249190890, precipitation: 2015-04
Station 249190090, temperature: 2015-01
Station 249190090, humidity: 2015-01
Station 249190090, precipitation: 2015-01
Station 249190440, temperature: 2015-01
Station 249190440, humidity: 2015-01
Station 249190440, precipitation: 2015-01
Station 249190240, temperature: 2015-01
Station 249190240, humidity: 2015-01
Station 249190240, precipitation: 2015-01
Station 249190480, temperature: 2015-01
Station 249190480, humidity: 2015-01
Station 249190480, precipitation: 2015-01
Station 249190190, temperature: 2015-01
Station 249190190, humidity: 2015-01
Station 249190190, precipitation: 2015-01
Station 249190130, temperature: 2015-01
Station 249190130, humidity: 2015-01
Station 249190130, precipitation: 2015-01
Station 249180210, temperature: 2015-01
Station 249180210, humidity: 2015-01
Station 249180210, precipitation: 2015-01
Station 2491903

In [44]:
df_weather_stations.temperature_start_date.value_counts()
df_weather_stations.precipitation_start_date.value_counts()

precipitation_start_date
2015-01    403
2019-01     56
2015-06     17
2015-04     14
2015-03      6
2022-01      5
2019-04      3
2015-12      1
2015-07      1
2020-02      1
2019-08      1
2019-06      1
2015-10      1
2015-05      1
2021-01      1
2019-10      1
2022-05      1
2019-12      1
2021-08      1
Name: count, dtype: int64

In [7]:
# Create boolean masks for each measurement type
combinations = pd.DataFrame({
    'temp_available': ~df_weather_stations.temperature_start_date.isna(),
    'precip_available': ~df_weather_stations.precipitation_start_date.isna(),
    'humidity_available': ~df_weather_stations.humidity_start_date.isna() if 'humidity_start_date' in df_weather_stations.columns else pd.Series(False, index=df_weather_stations.index)
})

# Create mutually exclusive categories
def categorize_station(row):
    t, p, h = row['temp_available'], row['precip_available'], row['humidity_available']
    if t and p and h:
        return 'All three (T+P+H)'
    elif t and p:
        return 'Temperature + Precipitation only'
    elif t and h:
        return 'Temperature + Humidity only'
    elif p and h:
        return 'Precipitation + Humidity only'
    elif t:
        return 'Temperature only'
    elif p:
        return 'Precipitation only'
    elif h:
        return 'Humidity only'
    else:
        return 'No data'

combinations['category'] = combinations.apply(categorize_station, axis=1)

# Count stations in each category
category_counts = combinations['category'].value_counts().reset_index()
category_counts.columns = ['Category', 'Count']

# Display results
display(category_counts)

# Summary total
print(f"\nTotal stations: {len(df_weather_stations)}")
print(f"Sum of all categories: {category_counts['Count'].sum()}")

,Category,Count
0,Precipitation only,263
1,All three (T+P+H),244
2,No data,209
3,Precipitation + Humidity only,8
4,Temperature + Humidity only,1
5,Temperature + Precipitation only,1



Total stations: 726
Sum of all categories: 726


In [8]:
def simplified_categorize_station(row):
    t, p, h = row['temp_available'], row['precip_available'], row['humidity_available']
    if t and p and h:
        return 'Precipitation, Temperature, Humidity'
    elif p and not t and not h:
        return 'Precipitation only'
    elif not t and not p and not h:
        return 'No data'
    else:
        return 'Other'

# Add simplified category to combinations dataframe
combinations['simplified_category'] = combinations.apply(simplified_categorize_station, axis=1)

# Add the category to the original dataframe
df_weather_stations['category'] = combinations['simplified_category']

# Display distribution of simplified categories
simplified_counts = df_weather_stations['category'].value_counts().reset_index()
simplified_counts.columns = ['Category', 'Count']
display(simplified_counts)

,Category,Count
0,Precipitation only,263
1,"Precipitation, Temperature, Humidity",244
2,No data,209
3,Other,10


In [9]:
df_weather_stations

,id,name,latitude,longitude,elevation,temperature_start_date,precipitation_start_date,humidity_start_date,category
0,249170080,Dzierżkowice,49.992222,17.846389,270.0,NaN,NaN,NaN,No data
1,249180010,Pszczyna,49.995556,18.919167,270.0,2015-01,2019-04,2015-01,"Precipitation, Temperature, Humidity"
2,249180020,Warszowice,49.991944,18.705556,270.0,NaN,2015-01,NaN,Precipitation only
3,249180030,Chałupki,49.925833,18.327500,198.0,NaN,NaN,NaN,No data
4,249180070,Mazańcowice,49.862500,18.969444,269.0,NaN,2015-01,NaN,Precipitation only
...,...,...,...,...,...,...,...,...,...
721,354180135,Hel,54.603611,18.811944,1.0,2015-01,2015-06,2015-01,"Precipitation, Temperature, Humidity"
722,354180155,Gdańsk-Świbno,54.333611,18.934444,7.0,2015-01,2019-01,2015-01,"Precipitation, Temperature, Humidity"
723,354190160,Elbląg-Milejewo,54.223333,19.543611,189.0,2015-01,2015-06,2015-01,"Precipitation, Temperature, Humidity"
724,354210185,KĘTRZYN,54.067222,21.366667,107.0,2015-01,2015-01,2015-01,"Precipitation, Temperature, Humidity"


In [21]:
# df1 = pd.read_csv('../../data/input/weather/20/01/B00300S_2015_01.csv', sep=';', encoding='windows-1250')
# df1

measurement_path = '../../data/input/weather/2022/01/B00300S_2022_01.csv'

columns = ['station_id', 'measurement', 'datetime', 'value']
df_measurements = pd.read_csv(measurement_path, sep=';', encoding='windows-1250', names=columns, header=None, index_col=False, dtype={'station_id': str, 'value': str, 'datetime': str, 'measurement': str})
df_measurements

,station_id,measurement,datetime,value
0,ď»ż249190890,B00300S,2022-01-01 00:00,"8,4"
1,249190890,B00300S,2022-01-01 00:10,"8,3"
2,249190890,B00300S,2022-01-01 00:20,"8,3"
3,249190890,B00300S,2022-01-01 00:30,"8,3"
4,249190890,B00300S,2022-01-01 00:40,"8,1"
...,...,...,...,...
1051184,252170390,B00300S,2022-01-31 23:10,"-2,4"
1051185,252170390,B00300S,2022-01-31 23:20,"-2,5"
1051186,252170390,B00300S,2022-01-31 23:30,"-2,7"
1051187,252170390,B00300S,2022-01-31 23:40,"-2,8"


In [24]:
df_weather_stations['id'].isin(df_measurements['station_id']).sum()

np.int64(238)